# Lab | Pandas - Solution Notebook

**Dataset**: Insurance company customer data

**Source**: https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

## Setup - Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv")
df.head()

---
## Challenge 1: Understanding the Data

### Step 1 - Identify the dimensions of the dataset

In [ ]:
# .shape returns (number of rows, number of columns)
df.shape

**Insight**: The dataset has **4008 rows** and **11 columns**.

### Step 2 - Determine data types and evaluate appropriateness

In [ ]:
# .info() shows column names, data types, and non-null counts
df.info()

**Insights on Data Types**:
- `Customer Lifetime Value` is stored as **object (string)** but should be **float** — it contains a `%` sign that needs to be removed
- `Number of Open Complaints` is stored as **object (string)** but should be **numeric** — values are formatted like `1/0/00` instead of just `0`
- All other columns appear to have appropriate data types

### Step 3 - Identify unique values and categorical vs numerical columns

In [ ]:
# .nunique() shows how many unique values each column has
df.nunique()

In [ ]:
# Check unique values for categorical columns
categorical_columns = ['ST', 'GENDER', 'Education', 'Policy Type', 'Vehicle Class']

for col in categorical_columns:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print()

In [ ]:
# Check range of numerical columns
numerical_columns = ['Income', 'Monthly Premium Auto', 'Total Claim Amount']

for col in numerical_columns:
    print(f"--- {col} ---")
    print(f"Min: {df[col].min()}")
    print(f"Max: {df[col].max()}")
    print()

**Insights**:
- **Categorical columns**: ST, GENDER, Education, Policy Type, Vehicle Class
- **Numerical columns**: Income, Monthly Premium Auto, Total Claim Amount
- The `ST` column has inconsistent values — both `Cali` and `California` represent the same state
- The `GENDER` column has inconsistent values — `F`, `Femal`, `female` should all be the same

### Step 4 - Summary statistics for numerical columns

In [ ]:
# .describe() gives mean, std, min, max, and quartiles for numerical columns
df.describe()

In [ ]:
# Calculate mean, median, and mode for numerical columns
numerical_columns = ['Income', 'Monthly Premium Auto', 'Total Claim Amount']

for col in numerical_columns:
    print(f"--- {col} ---")
    print(f"Mean:   {df[col].mean():.2f}")
    print(f"Median: {df[col].median():.2f}")
    print(f"Mode:   {df[col].mode()[0]:.2f}")
    print(f"Std:    {df[col].std():.2f}")
    print(f"Q25:    {df[col].quantile(0.25):.2f}")
    print(f"Q75:    {df[col].quantile(0.75):.2f}")
    print()

**Insights**:
- **Income**: Mean (~37,657) is higher than median (~33,889) suggesting right skew — some high earners pulling the average up
- **Monthly Premium Auto**: Most customers pay between 65-99 per month. The mode is 65 suggesting many customers are on the base plan
- **Total Claim Amount**: Wide range suggests significant variation in claim sizes

### Step 5 - Summary statistics for categorical columns

In [ ]:
# .describe(include='object') gives summary stats for categorical columns
df.describe(include='object')

**Insights**:
- **ST**: Oregon is the most common state
- **GENDER**: Female is the most common gender
- **Education**: Bachelor is the most common education level
- **Policy Type**: Personal Auto is the most common policy
- **Vehicle Class**: Four-Door Car is the most common vehicle class

---
## Challenge 2: Analyzing the Data

### Exercise 1 - Top 5 least common customer locations

In [ ]:
# Step 1: Create a Series with location frequencies using value_counts()
# value_counts() counts how many times each value appears
location_frequencies = df['ST'].value_counts()

# Step 2: Sort in ascending order (least common first) and take top 5
# ascending=True → smallest counts first
# .head(5) → take only the first 5
top_5_least_common = location_frequencies.sort_values(ascending=True).head(5)

print("Top 5 least common customer locations:")
print(top_5_least_common)

**Insight**: The 5 least common customer locations are shown above. Note that `AZ` and `Arizona` likely represent the same state — this is a data quality issue that would need cleaning.

### Exercise 2 - Total policies sold per policy type and highest selling type

In [ ]:
# Step 1: Create a Series with policy type frequencies
# value_counts() counts how many times each policy type appears = total policies sold
policy_counts = df['Policy Type'].value_counts()

print("Total policies sold per type:")
print(policy_counts)
print()

# Step 2: Find the policy type with the highest number of policies sold
# .idxmax() returns the INDEX (label) of the maximum value
best_selling_policy = policy_counts.idxmax()

print(f"Policy type with highest sales: {best_selling_policy}")
print(f"Total policies sold: {policy_counts.max()}")

**Insight**: **Personal Auto** is the most sold policy type. This suggests the majority of customers are individual car owners rather than corporate clients.

### Exercise 3 - Average income comparison between Personal Auto and Corporate Auto

In [ ]:
# Step 1: Use .loc to create two separate DataFrames
# .loc filters rows based on a condition
personal_auto_df = df.loc[df['Policy Type'] == 'Personal Auto']
corporate_auto_df = df.loc[df['Policy Type'] == 'Corporate Auto']

# Step 2: Calculate average income for each policy type
avg_income_personal = personal_auto_df['Income'].mean()
avg_income_corporate = corporate_auto_df['Income'].mean()

# Step 3: Print the results
print(f"Average Income - Personal Auto:  ${avg_income_personal:,.2f}")
print(f"Average Income - Corporate Auto: ${avg_income_corporate:,.2f}")
print()

difference = avg_income_corporate - avg_income_personal
print(f"Difference: ${difference:,.2f} (Corporate Auto customers earn more)" if difference > 0 else f"Difference: ${abs(difference):,.2f} (Personal Auto customers earn more)")

**Insight**: Compare the two averages — this tells the sales team whether Personal Auto customers tend to have lower income than Corporate Auto customers, which can inform marketing and pricing strategies.

### Bonus Exercise 4 - Customers with high policy claim amounts (top 25%)

In [ ]:
# Step 1: Review statistics for Total Claim Amount
print("Summary statistics for Total Claim Amount:")
print(df['Total Claim Amount'].describe())
print()

In [ ]:
# Step 2: Find the 75th percentile (top 25% threshold)
# .quantile(0.75) gives the value below which 75% of the data falls
# Anything ABOVE this value is in the top 25%
percentile_75 = df['Total Claim Amount'].quantile(0.75)
print(f"75th percentile of Total Claim Amount: {percentile_75:.2f}")
print(f"This means 75% of customers have claims below ${percentile_75:.2f}")
print(f"We want to focus on customers with claims ABOVE this value")
print()

In [ ]:
# Step 3: Create a DataFrame with only high claim amount customers
# Filter rows where Total Claim Amount is greater than the 75th percentile
high_claim_df = df[df['Total Claim Amount'] > percentile_75]

print(f"Number of high claim customers: {len(high_claim_df)}")
print(f"This represents {len(high_claim_df)/len(df)*100:.1f}% of all customers")
print()

In [ ]:
# Step 4: Calculate summary statistics for high claim customers
print("Summary statistics for high claim amount customers:")
print(high_claim_df.describe())

**Insight**: Customers in the top 25% of claim amounts represent a key segment for the insurance company. These customers cost the company more in claims and may require special attention for:
- Risk assessment and pricing adjustments
- Customer retention strategies
- Fraud detection review